In [1]:
import os

# HF_HUB_DISABLE_XET is an environment variable for the Hugging Face Hub Python library 
# that disables the default high-performance Xet (hf-xet) download backend and falls 
# back to standard HTTP/LFS methods
os.environ["HF_HUB_DISABLE_XET"] = "1" 

# # Disable GCE metadata lookup
os.environ["NO_GCE_CHECK"] = "true"
os.environ["GCE_METADATA_TIMEOUT"] = "1"

# # Force TensorFlow's gcs filesystem to use HTTP/1.1 instead of HTTP/2
# os.environ["TF_GCS_USE_HTTP1"] = "true"

# # Tune GCS timeouts and block sizes to avoid hanging connections
# os.environ["TF_GCS_READ_CACHE_DISABLED"] = "1"

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from transformers import AutoModelForVision2Seq, AutoProcessor
from PIL import Image, ImageDraw
import tensorflow_datasets as tfds

import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import h5py

2026-08-25 09:29:15.157769: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-25 09:29:15.198137: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-08-25 09:29:15.198165: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-08-25 09:29:15.199097: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-25 09:29:15.205119: I tensorflow/core/platform/cpu_feature_guar

#### Check GPU

In [3]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory allocated:", torch.cuda.memory_allocated() / 1024**2, "MB")
    print("GPU memory reserved:", torch.cuda.memory_reserved() / 1024**2, "MB")
    print("Device properties:", torch.cuda.get_device_properties(0))

PyTorch: 2.2.0+cu121
CUDA available: True
GPU: Quadro RTX 5000
GPU memory allocated: 0.0 MB
GPU memory reserved: 0.0 MB
Device properties: _CudaDeviceProperties(name='Quadro RTX 5000', major=7, minor=5, total_memory=15929MB, multi_processor_count=48)


In [2]:
# Load a small snippet of Bridge V2 from Open X-Embodiment
ds = tfds.load("bridge", split="train[:1]", data_dir="gs://gresearch/robotics")

# for episode in ds:
#     for step in episode['steps']:
#         raw_image_np = step['observation']['image'].numpy()
#         image = Image.fromarray(raw_image_np)
#         break

2026-08-25 09:30:12.607834: I external/local_tsl/tsl/platform/cloud/google_auth_provider.cc:180] Attempting an empty bearer token since no token was retrieved from files, and GCE metadata check was skipped.
2026-08-25 09:31:13.672807: E external/local_tsl/tsl/platform/cloud/curl_http_request.cc:610] The transmission  of request 0x557ef4c57660 (URI: https://www.googleapis.com/storage/v1/b/gresearch/o/robotics%2Fbridge%2F.config%2Fmetadata.json?fields=size%2Cgeneration%2Cupdated) has been stuck at 0 of 0 bytes for 61 seconds and will be aborted. CURL timing information: lookup time: 0.003674 (No error), connect time: 0 (No error), pre-transfer time: 0 (No error), start-transfer time: 0 (No error)
2026-08-25 09:31:14.981468: I external/local_tsl/tsl/platform/cloud/google_auth_provider.cc:180] Attempting an empty bearer token since no token was retrieved from files, and GCE metadata check was skipped.
2026-08-25 09:32:15.039710: E external/local_tsl/tsl/platform/cloud/curl_http_request.cc:

AbortedError: All 10 retry attempts failed. The last failure: Error executing an HTTP request: libcurl code 42 meaning 'Operation was aborted by an application callback', error details: Callback aborted
	 when reading metadata of gs://gresearch/robotics/bridge/.config/metadata.json

#### OpenVLA Toolbox Example

In [ ]:
# 1. Configure 4-bit quantization to fit within ~7 GB of VRAM
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

# Load Processor & VLA
processor = AutoProcessor.from_pretrained("openvla/openvla-7b", trust_remote_code=True)
vla = AutoModelForVision2Seq.from_pretrained(
    "openvla/openvla-7b", 
    attn_implementation="eager",  # [Optional] Requires `flash_attn`
    torch_dtype=torch.float16, # from torch.bfloat16 to torch.float16: Quadro RTX 5000 is a Turing GPU and supports hardware acceleration for float16.
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    quantization_config=quantization_config
).to("cuda:0")


/user/fiorenza/u24781/.conda/envs/openvla/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/user/fiorenza/u24781/.conda/envs/openvla/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


/user/fiorenza/u24781/.conda/envs/openvla/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model-00001-of-00003.safetensors:   0%|          | 0.00/6.95G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/6.97G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.16G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

NameError: name 'get_from_camera' is not defined

In [10]:
# Grab image input & format prompt
# Synthetic placeholder: a simple scene with shapes standing in for objects,
image = Image.new("RGB", (256, 256), color=(200, 200, 200))
draw = ImageDraw.Draw(image)
draw.rectangle([80, 120, 140, 180], fill=(200, 30, 30))   # red "block"
draw.ellipse([160, 140, 200, 180], fill=(30, 30, 200))    # blue "object"

# image: Image.Image = get_from_camera(...) # you need an image here.
prompt = "In: What action should the robot take to grab the red square?\nOut:"

# # Predict Action (7-DoF; un-normalize for BridgeData V2)
inputs = processor(prompt, image).to("cuda:0", dtype=torch.float16)
action = vla.predict_action(**inputs, unnorm_key="bridge_orig", do_sample=False)

# # Execute...
# robot.act(action, ...)

RuntimeError: The size of tensor a (275) must match the size of tensor b (274) at non-singleton dimension 3